First, turn your Matlab session as a shared engine by using:
```MATLAB
matlab.engine.shareEngine('TestEngine')
```

Then, find and start this shared Matlab engine

In [ ]:
import os
import matlab.engine


def use_default_engine():
    eng = None
    available_engines = matlab.engine.find_matlab()
    print(f"Available engines: {available_engines}")

    if type(available_engines) == tuple:
        if len(available_engines) > 0:
            eng = matlab.engine.connect_matlab(available_engines[0])

    return eng


eng = use_default_engine()

Available engines: ('TestEngine',)


Test an embedded function

In [15]:
res = eng.sqrt(matlab.double(16))
print(res, type(res))

4.0 <class 'float'>


Test our own function, which must be added to the path of the engine if it is not already in the Matlab path

In [16]:
this_path = os.path.abspath("")
file_path = os.path.join(this_path, r"matlab_code")

eng.addpath(file_path, nargout=0)
eng.cd(this_path, nargout=0)

s, p = eng.my_square(matlab.double(3), nargout=2)
print(s, p)

9.0 True


Change the path to the current directory and run the script

In [17]:
eng.cd(this_path, nargout=0)
eng.run('matlab_code/script.m', nargout=0)

Access workspace variables and change them

In [18]:
print(type(eng.workspace))
print(eng.workspace['x'])
print(eng.workspace['y'])

eng.workspace['y'] = 2.0
print(eng.workspace['y'])

<class 'matlab.engine.matlabengine.MatlabWorkSpace'>
16.0
4.0
2.0


Redirect errors outputs to Python

In [19]:
import io
out = io.StringIO()
err = io.StringIO()
ret = eng.dec2base(2**60,16,stdout=out,stderr=err)

print(err.getvalue())

MatlabExecutionError: 
  File C:\Program Files\MATLAB\R2024b\toolbox\matlab\strfun\dec2base.m, line 10, in dec2base
First argument must be an array of integers, 0 <= D <= flintmax.


In [28]:
print(err.getvalue())

Error using dec2base (line 10)
First argument must be an array of integers, 0 <= D <= flintmax.




Handle objects and call them from Python

In [20]:
tr = eng.Triangle(5.0,3.0)
a = eng.area(tr)
print(a)

7.5


In [21]:
eng.workspace["wtr"] = tr
b = eng.eval("wtr.Base")
print(b)

5.0


In [22]:
eng.setHeight(tr,8.0,nargout=0)
a = eng.area(tr)
print(a)

20.0


Close the Matlab engine

In [23]:
eng.quit()